In [4]:
!pip install agentpy -q

import pandas as pd
import agentpy as ap
import random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

num_agents = int(input("Enter number of agents (e.g., 5): "))
grid_size = int(input("Enter grid size (e.g., 10 for 10x10): "))
num_steps = int(input("Enter number of steps (e.g., 20): "))

class RandomWalker(ap.Agent):
    def setup(self):
        # store movement history so we can draw the trails later
        self.path = []

    def step(self):
        # tweaking the rules: making them prefer moving right (50% chance)
        # instead of the standard equal distribution just to see the pattern change
        directions = [(1,0), (-1,0), (0,1), (0,-1)]
        weights = [50, 16.6, 16.6, 16.6]
        direction = random.choices(directions, weights=weights, k=1)[0]

        x, y = self.position

        # clamp coordinates so they don't walk off the grid bounds
        new_x = max(0, min(self.model.p.grid_size[0]-1, x + direction[0]))
        new_y = max(0, min(self.model.p.grid_size[1]-1, y + direction[1]))

        self.position = (new_x, new_y)

        # log the new spot to the path tracker
        self.path.append(self.position)

class RandomWalkModel(ap.Model):
    def setup(self):
        self.agents = ap.AgentList(self, self.p.agents, RandomWalker)

        # drop agents at random starting points
        for agent in self.agents:
            agent.position = (random.randint(0, self.p.grid_size[0]-1),
                              random.randint(0, self.p.grid_size[1]-1))
            agent.path.append(agent.position) # need to log the spawn point too

        self.grid = ap.Grid(self, self.p.grid_size, torus=False)
        self.grid.add_agents(self.agents)

    def step(self):
        for agent in self.agents:
            agent.step()

# load params
parameters = {
    'agents': num_agents,
    'grid_size': (grid_size, grid_size),
    'steps': num_steps
}

# init and setup model
model = RandomWalkModel(parameters)
model.setup()

# plotting stuff
fig, ax = plt.subplots(figsize=(6, 6))

def update(frame):
    ax.clear()
    ax.set_xlim(-0.5, model.p.grid_size[0] - 0.5)
    ax.set_ylim(-0.5, model.p.grid_size[1] - 0.5)
    ax.set_xticks(range(model.p.grid_size[0]))
    ax.set_yticks(range(model.p.grid_size[1]))
    ax.grid(True)
    ax.set_title(f"Random Walk Simulation - Step {frame + 1}")

    model.step()

    # draw the history trails
    for agent in model.agents:
        if len(agent.path) > 1:
            px, py = zip(*agent.path)
            ax.plot(px, py, linestyle='-', alpha=0.4)

    # draw the actual agents over the trails
    x = [agent.position[0] for agent in model.agents]
    y = [agent.position[1] for agent in model.agents]
    scat = ax.scatter(x, y, s=200, c='red', edgecolors='black')

    return scat,

ani = animation.FuncAnimation(fig, update, frames=model.p.steps, blit=False, repeat=False)
plt.close() # keep it from rendering a weird static duplicate below the anim

# display the animation
display(HTML(ani.to_jshtml()))

# print out the final coords just to check where everyone ended up
print("\n--- Final Agent Positions ---")
for i, agent in enumerate(model.agents):
    print(f"Agent {i+1} Final Position: {agent.position}")

# Gather the final data
final_data = []
for i, agent in enumerate(model.agents):
    final_data.append({
        "Agent_ID": i + 1,
        "X_Coordinate": agent.position[0],
        "Y_Coordinate": agent.position[1]
    })

df = pd.DataFrame(final_data)

# Dynamically name the file based on the user inputs
csv_filename = f"scenario_{num_agents}agents_{grid_size}x{grid_size}_steps{num_steps}.csv"

# Export to CSV
df.to_csv(csv_filename, index=False)
print(f"\n✅ Exported final positions to: {csv_filename}")

Enter number of agents (e.g., 5): 40
Enter grid size (e.g., 10 for 10x10): 8
Enter number of steps (e.g., 20): 25



--- Final Agent Positions ---
Agent 1 Final Position: (7, 5)
Agent 2 Final Position: (6, 6)
Agent 3 Final Position: (7, 2)
Agent 4 Final Position: (7, 7)
Agent 5 Final Position: (7, 0)
Agent 6 Final Position: (6, 5)
Agent 7 Final Position: (6, 6)
Agent 8 Final Position: (6, 0)
Agent 9 Final Position: (7, 5)
Agent 10 Final Position: (6, 2)
Agent 11 Final Position: (7, 4)
Agent 12 Final Position: (7, 6)
Agent 13 Final Position: (5, 3)
Agent 14 Final Position: (7, 1)
Agent 15 Final Position: (6, 4)
Agent 16 Final Position: (7, 1)
Agent 17 Final Position: (7, 1)
Agent 18 Final Position: (4, 2)
Agent 19 Final Position: (6, 3)
Agent 20 Final Position: (7, 4)
Agent 21 Final Position: (7, 4)
Agent 22 Final Position: (5, 3)
Agent 23 Final Position: (7, 7)
Agent 24 Final Position: (7, 0)
Agent 25 Final Position: (5, 5)
Agent 26 Final Position: (7, 4)
Agent 27 Final Position: (7, 0)
Agent 28 Final Position: (6, 4)
Agent 29 Final Position: (7, 1)
Agent 30 Final Position: (7, 3)
Agent 31 Final Pos